In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H19 — Wave-Based Multi-Engine SAT Solver with Basin Clustering
# ══════════════════════════════════════════════════════════════════════
#
# ARCHITECTURE (per instance):
#   repeat until solved or time/wave budget exhausted:
#     1. K = K_BASE × (1 + floor(stagnation/3))    → adaptive engine count
#     2. Generate K diverse seeds (5 types)          → diverse starting points
#     3. Each engine: P particles around seed        → shape [K, P, n]
#     4. Run K gravity engines (GPU batch, independent)
#     5. Extract best particle per engine            → K candidates
#     6. Stage 1: reject if unsat/n > 20%            → loose filter
#     7. Stage 2: rank, keep top 32                  → score-based selection
#     8. Basin clustering (Hamming)                  → distinct representatives
#     9. Selective BPR on basin reps                 → adaptive flip budget
#    10. Update best_model + stagnation
#
# KEY INNOVATIONS over H15:
#   • Multi-engine (K independent gravity runs) vs single gravity run
#   • Diverse seeding: local/inverse/partial-flip/random/mask-XOR
#   • Basin clustering: Hamming-based → no wasted BPR on same basin
#   • Selective BPR: budget ∝ proximity (60K/20K/5K by closeness)
#   • Anti-collapse: push particles from mean every 50 steps
#   • Micro-reset: flip ~2% of variables every 200 steps
#   • Adaptive K: grows with stagnation (wider search when stuck)
#   • Fused GPU extraction: discretize+eval+extract ON GPU (H19++)
#
# Author: Odeyemi Olusegun Israel
# ══════════════════════════════════════════════════════════════════════
import torch, numpy as np, time, math, os
from numba import njit
from concurrent.futures import ThreadPoolExecutor, as_completed

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

N_WORKERS = min(os.cpu_count() or 2, 8)

# ═══════ AUTO-SCALE BASED ON GPU VRAM ═══════
if device.type == 'cuda':
    _vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    if _vram >= 70:      # A100-80GB / RTX PRO 6000
        K_BASE = 16;  ENGINE_BATCH = 60;  P_ENGINE = 250
    elif _vram >= 35:    # A100-40GB
        K_BASE = 12;  ENGINE_BATCH = 48;  P_ENGINE = 200
    elif _vram >= 14:    # T4 / V100
        K_BASE = 8;   ENGINE_BATCH = 32;  P_ENGINE = 200
    else:
        K_BASE = 4;   ENGINE_BATCH = 16;  P_ENGINE = 150
else:
    K_BASE = 4;  ENGINE_BATCH = 4;  P_ENGINE = 100

# ═══════ H19 ARCHITECTURE PARAMETERS ═══════
STEPS_ENGINE   = 2000     # gravity steps per engine
MAX_WAVES      = 6        # max wave iterations per config
MAX_BASINS     = 6        # max distinct basins to send to BPR
TOP_CANDIDATES = 32       # max candidates after stage 2

# Stage 1 filter
UNSAT_REJECT_FRAC = 0.20  # reject if unsat > 0.20 × n

# Basin clustering
HAMMING_TAU_FRAC = 0.15   # cluster merge threshold = 0.15 × n

# BPR selective: (unsat_frac_of_m, run_probability, flip_budget)
BPR_TIERS = [
    (0.005, 1.0,  60000),   # < 0.5%:  always, 60K
    (0.01,  1.0,  40000),   # < 1.0%:  always, 40K
    (0.03,  0.50, 20000),   # < 3.0%:  50%, 20K
    (1.00,  0.10,  5000),   # else:    10%, 5K
]

# Stagnation scaling
STAG_SCALE = 3            # K factor grows every 3 stagnant waves
K_GPU_LIMIT = 64          # cap on engine count

# BPR chain parameters
CHAIN_PATIENCE  = 5000
BRANCH_PATIENCE = 80
COOL_RATE       = 0.95
STUB_FRAC       = 0.08
WEIGHT_BUMP     = 2.0
WEIGHT_DECAY    = 0.9
BPR_BETA        = 0.3


# ══════════════════════════════════════════════════════════════════════
#  INSTANCE GENERATOR
# ══════════════════════════════════════════════════════════════════════

def generate_3sat_instance(n, m):
    vars_idx = torch.randint(0, n, (m, 3))
    signs = torch.randint(0, 2, (m, 3)) * 2 - 1
    return list(zip(vars_idx.tolist(), signs.tolist()))


# ══════════════════════════════════════════════════════════════════════
#  TREE-WALK BPR — chain death + stubbornness
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def bpr_chain(clauses_v, clauses_s, assignment, weight,
              max_flips=200000, T_init=0.5, T_min=0.01,
              p_random=0.1, beta=0.3,
              chain_patience=5000, branch_patience=80,
              cool_rate=0.95, stub_frac=0.08,
              weight_bump=2.0, weight_decay=0.9):
    m = clauses_v.shape[0]
    n = assignment.shape[0]
    n_stub = max(2, int(stub_frac * n))

    var_count = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            var_count[clauses_v[c, j]] += 1
    var_off = np.zeros(n + 1, dtype=np.int32)
    for vi in range(n):
        var_off[vi + 1] = var_off[vi] + var_count[vi]
    var_adj = np.zeros(var_off[n], dtype=np.int32)
    var_sign = np.zeros(var_off[n], dtype=np.int32)
    fill = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            vi = clauses_v[c, j]
            pos = var_off[vi] + fill[vi]
            var_adj[pos] = c
            var_sign[pos] = clauses_s[c, j]
            fill[vi] += 1

    clause_w = np.ones(m, dtype=np.float64)
    clause_sat = np.zeros(m, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            vi = clauses_v[c, j]
            s_ = clauses_s[c, j]
            if (assignment[vi] == 1 and s_ == 1) or (assignment[vi] == 0 and s_ == -1):
                clause_sat[c] += 1

    unsat_list = np.zeros(m, dtype=np.int32)
    unsat_pos = np.full(m, -1, dtype=np.int32)
    n_unsat = 0
    for c in range(m):
        if clause_sat[c] == 0:
            unsat_pos[c] = n_unsat
            unsat_list[n_unsat] = c
            n_unsat += 1

    best_n_unsat = n_unsat
    best_assign = assignment.copy()
    chain_id = 0
    chain_stale = 0
    chain_best_unsat = n_unsat
    in_branch = False
    branch_stale = 0
    recent_size = 8
    recent_flipped = np.full(recent_size, -1, dtype=np.int32)
    recent_idx = 0
    stubbornness = np.zeros(n, dtype=np.float64)

    for flip in range(max_flips):
        if n_unsat == 0:
            return assignment, flip, chain_id + 1, 0

        if n_unsat < best_n_unsat:
            best_n_unsat = n_unsat
            for i in range(n):
                best_assign[i] = assignment[i]
            chain_stale = 0
            chain_best_unsat = n_unsat
        elif n_unsat < chain_best_unsat:
            chain_best_unsat = n_unsat
            chain_stale = 0
        else:
            chain_stale += 1

        if chain_stale >= chain_patience and n_unsat > 0:
            for ui in range(n_unsat):
                clause_w[unsat_list[ui]] += weight_bump
            for c in range(m):
                clause_w[c] *= weight_decay
            for vi in range(n):
                stubbornness[vi] = 0.0
            for ui in range(n_unsat):
                cc = unsat_list[ui]
                w_c = clause_w[cc]
                for j in range(3):
                    stubbornness[clauses_v[cc, j]] += w_c
            stub_to_flip = np.zeros(n_stub, dtype=np.int32)
            stub_used = np.zeros(n, dtype=np.int8)
            for k in range(n_stub):
                best_sv = -1.0
                best_vi = 0
                for vi in range(n):
                    if stub_used[vi] == 0 and stubbornness[vi] > best_sv:
                        best_sv = stubbornness[vi]
                        best_vi = vi
                stub_to_flip[k] = best_vi
                stub_used[best_vi] = 1
            for i in range(n):
                assignment[i] = best_assign[i]
            for k in range(n_stub):
                assignment[stub_to_flip[k]] = 1 - assignment[stub_to_flip[k]]
            n_random = max(1, n // 100)
            for _ in range(n_random):
                vi = np.random.randint(n)
                if np.random.random() < 0.3:
                    assignment[vi] = 1 - assignment[vi]

            n_unsat = 0
            for c in range(m):
                clause_sat[c] = 0
                for j in range(3):
                    vi = clauses_v[c, j]
                    s_ = clauses_s[c, j]
                    if (assignment[vi] == 1 and s_ == 1) or (assignment[vi] == 0 and s_ == -1):
                        clause_sat[c] += 1
                if clause_sat[c] == 0:
                    unsat_pos[c] = n_unsat
                    unsat_list[n_unsat] = c
                    n_unsat += 1
                else:
                    unsat_pos[c] = -1

            if n_unsat == 0:
                return assignment, flip, chain_id + 1, 0
            chain_id += 1
            chain_stale = 0
            chain_best_unsat = n_unsat
            in_branch = False
            branch_stale = 0
            for ri in range(recent_size):
                recent_flipped[ri] = -1
            recent_idx = 0
            continue

        ci = -1
        if in_branch and branch_stale < branch_patience:
            best_neighbor_w = -1.0
            for ri in range(recent_size):
                rv = recent_flipped[ri]
                if rv < 0:
                    continue
                for idx in range(var_off[rv], var_off[rv + 1]):
                    cc = var_adj[idx]
                    if clause_sat[cc] == 0 and clause_w[cc] > best_neighbor_w:
                        best_neighbor_w = clause_w[cc]
                        ci = cc
            if ci < 0:
                in_branch = False

        if not in_branch or ci < 0:
            ci = unsat_list[np.random.randint(n_unsat)]
            in_branch = True
            branch_stale = 0
            for ri in range(recent_size):
                recent_flipped[ri] = -1
            recent_idx = 0

        T_max_chain = T_init * (cool_rate ** min(chain_id, 30))
        local_prog = min(1.0, chain_stale / chain_patience)
        T = T_min + (T_max_chain - T_min) * 0.5 * (1.0 + np.cos(3.141592653589793 * local_prog))

        if np.random.random() < p_random:
            v_flip = clauses_v[ci, np.random.randint(3)]
        else:
            int_brks = np.zeros(3, dtype=np.int32)
            scores = np.zeros(3, dtype=np.float64)
            for j in range(3):
                v_cand = clauses_v[ci, j]
                i_brk = 0; w_brk = 0.0; w_make = 0.0
                for idx in range(var_off[v_cand], var_off[v_cand + 1]):
                    cc2 = var_adj[idx]
                    s_here = var_sign[idx]
                    satisfies = ((assignment[v_cand] == 1 and s_here == 1) or
                                 (assignment[v_cand] == 0 and s_here == -1))
                    if satisfies:
                        if clause_sat[cc2] == 1:
                            i_brk += 1
                            w_brk += clause_w[cc2]
                    else:
                        if clause_sat[cc2] == 0:
                            w_make += clause_w[cc2]
                int_brks[j] = i_brk
                delta = w_brk - w_make
                scores[j] = np.exp(-delta / (T + 1e-10)) * (1.0 + beta * weight[v_cand])

            zb_n = 0
            zb_opts = np.zeros(3, dtype=np.int32)
            for j in range(3):
                if int_brks[j] == 0:
                    zb_opts[zb_n] = j
                    zb_n += 1
            if zb_n > 0:
                v_flip = clauses_v[ci, zb_opts[np.random.randint(zb_n)]]
            else:
                total = scores[0] + scores[1] + scores[2]
                if total < 1e-30:
                    v_flip = clauses_v[ci, np.random.randint(3)]
                else:
                    r = np.random.random() * total
                    if r <= scores[0]:
                        v_flip = clauses_v[ci, 0]
                    elif r <= scores[0] + scores[1]:
                        v_flip = clauses_v[ci, 1]
                    else:
                        v_flip = clauses_v[ci, 2]

        old_n_unsat = n_unsat
        assignment[v_flip] = 1 - assignment[v_flip]
        for idx in range(var_off[v_flip], var_off[v_flip + 1]):
            cc2 = var_adj[idx]
            s_here = var_sign[idx]
            old_sat = clause_sat[cc2]
            now_satisfies = ((assignment[v_flip] == 1 and s_here == 1) or
                             (assignment[v_flip] == 0 and s_here == -1))
            if now_satisfies:
                clause_sat[cc2] += 1
            else:
                clause_sat[cc2] -= 1
            new_sat = clause_sat[cc2]
            if old_sat == 0 and new_sat > 0:
                pos = unsat_pos[cc2]
                last = unsat_list[n_unsat - 1]
                unsat_list[pos] = last
                unsat_pos[last] = pos
                unsat_pos[cc2] = -1
                n_unsat -= 1
            elif old_sat > 0 and new_sat == 0:
                unsat_list[n_unsat] = cc2
                unsat_pos[cc2] = n_unsat
                n_unsat += 1

        if n_unsat < old_n_unsat:
            branch_stale = 0
        else:
            branch_stale += 1
        recent_flipped[recent_idx % recent_size] = v_flip
        recent_idx += 1

    return best_assign, max_flips, chain_id + 1, best_n_unsat


# ══════════════════════════════════════════════════════════════════════
#  NUMBA SAT / UNSAT
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def check_sat(clauses_v, clauses_s, assignment):
    m = clauses_v.shape[0]
    for c in range(m):
        sat = False
        for j in range(3):
            vi = clauses_v[c, j]
            si = clauses_s[c, j]
            if (assignment[vi] == 1 and si == 1) or (assignment[vi] == 0 and si == -1):
                sat = True
                break
        if not sat:
            return False
    return True

@njit(cache=True)
def count_unsat(clauses_v, clauses_s, assignment):
    m = clauses_v.shape[0]
    cnt = 0
    for c in range(m):
        sat = False
        for j in range(3):
            vi = clauses_v[c, j]
            si = clauses_s[c, j]
            if (assignment[vi] == 1 and si == 1) or (assignment[vi] == 0 and si == -1):
                sat = True
                break
        if not sat:
            cnt += 1
    return cnt


# ══════════════════════════════════════════════════════════════════════
#  BATCHED ENERGY
# ══════════════════════════════════════════════════════════════════════

def _energy_batched(s, mu_val, vars_t, signs_t):
    B, P, n = s.shape
    m = vars_t.shape[1]
    prod_val = torch.ones(B, P, m, device=s.device, dtype=s.dtype)
    for j in range(3):
        idx_j = vars_t[:, :, j].unsqueeze(1).expand(B, P, m)
        gathered_j = torch.gather(s, 2, idx_j)
        lit_j = gathered_j * signs_t[:, None, :, j]
        prod_val = prod_val * (1.0 - lit_j)
    e_sat = (prod_val / 8.0).sum(dim=-1)
    if mu_val > 0:
        return e_sat + mu_val * ((1.0 - s * s) ** 2).sum(dim=-1)
    return e_sat


# ══════════════════════════════════════════════════════════════════════
#  GRAVITY CORE — H19 (anti-collapse + micro-reset)
# ══════════════════════════════════════════════════════════════════════

def _gravity_core_h19(s, vars_t, signs_t, steps, lr=0.05,
                      momentum_beta=0.9, mu_scale=0.1,
                      G_max=0.10, top_k_frac=0.1,
                      gravity_start=0.2, elite_repulsion=0.5,
                      gravity_interval=20):
    B, particles, n = s.shape
    gi = gravity_interval
    use_amp = (device.type == 'cuda')

    vel = torch.zeros_like(s)
    grav_step = int(gravity_start * steps)
    delay_step = int(0.7 * steps)
    top_k = max(1, int(top_k_frac * particles))
    theta = None

    best_e = torch.full((B, particles), float('inf'), device=device)
    plateau_count = torch.zeros(B, particles, device=device)
    decay_arr = 1.0 / (1.0 + 0.002 * torch.arange(
        steps, device=device, dtype=torch.float32))
    cached_targets = [None] * B

    for step in range(steps):
        if step < delay_step:
            mu_val = 0.0
        else:
            t_l = (step - delay_step) / (steps - delay_step)
            mu_val = mu_scale * 0.5 * (1.0 - math.cos(math.pi * t_l))

        s = s.detach().requires_grad_(True)
        if use_amp:
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                e = _energy_batched(s, mu_val, vars_t, signs_t)
                e_f32 = e.float()
        else:
            e_f32 = _energy_batched(s, mu_val, vars_t, signs_t)

        e_vals = e_f32.detach()
        e_f32.sum().backward()
        g = s.grad.detach().clone()

        with torch.no_grad():
            decay = decay_arr[step]
            improved = e_vals < best_e
            best_e = torch.where(improved, e_vals, best_e)
            plateau_count = torch.where(improved,
                torch.zeros_like(plateau_count), plateau_count + 1)
            pm = (plateau_count >= 50).unsqueeze(2)

            gnorm = g.norm(dim=2, keepdim=True).clamp_(min=1e-10)
            dte = (lr * decay) / (1.0 + 0.05 * gnorm)
            dte = dte * torch.where(pm, torch.tensor(2.0, device=device),
                                         torch.tensor(1.0, device=device))

            if theta is None:
                theta = float(e_vals.median()) + 1e-8
            damp = 1.0 / (1.0 + e_vals.unsqueeze(2) / theta)
            gam = (e_vals.clamp(min=0) / (e_vals + 1.0)).unsqueeze(2)

            vel = momentum_beta * vel - dte * damp * (1.0 + gam) * g

            ns_base = 0.03 * decay
            noise = torch.randn_like(s) * torch.where(pm, 4.0 * ns_base, ns_base)
            s = (s + vel + noise).clamp_(-1, 1)

            # Anti-collapse every 50 steps
            if (step + 1) % 50 == 0:
                s_mean = s.mean(dim=1, keepdim=True)
                s = s + 0.05 * (s - s_mean)
                s.clamp_(-1, 1)

            # Micro-reset every 200 steps
            if (step + 1) % 200 == 0:
                flip_mask = torch.rand(B, particles, n, device=device) < 0.02
                s = torch.where(flip_mask, -s, s)

            # Elite gravity
            if step >= grav_step and (step - grav_step) % gi == 0:
                progress = (step - grav_step) / (steps - grav_step)
                g_mag = G_max * progress * progress * gi
                for b in range(B):
                    _, top_idx = e_vals[b].topk(top_k, largest=False)
                    elite = s[b, top_idx]
                    d = torch.cdist(s[b].unsqueeze(0), elite.unsqueeze(0))[0]
                    cached_targets[b] = elite[d.argmin(dim=1)]
                    if top_k > 1:
                        ed = torch.cdist(elite.unsqueeze(0), elite.unsqueeze(0))[0]
                        ed.fill_diagonal_(float('inf'))
                        nn_e = ed.argmin(dim=1)
                        push = elite - elite[nn_e]
                        pn = push.norm(dim=1, keepdim=True).clamp_(min=1e-6)
                        s[b, top_idx] += (elite_repulsion * g_mag) * (push / pn)
                    s[b].add_(g_mag * (cached_targets[b] - s[b]))
            elif step >= grav_step:
                for b in range(B):
                    if cached_targets[b] is not None:
                        progress = (step - grav_step) / (steps - grav_step)
                        g_mag = G_max * progress * progress
                        s[b].add_(g_mag * (cached_targets[b] - s[b]))

            s.clamp_(-1, 1)
            if (step + 1) % 200 == 0:
                theta = float(e_vals.median()) + 1e-8

    return s.detach()


# ══════════════════════════════════════════════════════════════════════
#  DIVERSE SEED GENERATION (5 types)
# ══════════════════════════════════════════════════════════════════════

def generate_diverse_seeds(K, n, best_model=None):
    seeds = np.zeros((K, n), dtype=np.int32)
    if best_model is None:
        for k in range(K):
            seeds[k] = np.random.randint(0, 2, n).astype(np.int32)
        return seeds

    for k in range(K):
        r = np.random.random()
        if r < 0.30:
            seeds[k] = best_model.copy()
            nf = max(1, int(0.05 * n))
            idx = np.random.choice(n, nf, replace=False)
            seeds[k][idx] = 1 - seeds[k][idx]
        elif r < 0.50:
            seeds[k] = 1 - best_model
        elif r < 0.70:
            seeds[k] = best_model.copy()
            frac = 0.20 + 0.10 * np.random.random()
            nf = max(1, int(frac * n))
            idx = np.random.choice(n, nf, replace=False)
            seeds[k][idx] = 1 - seeds[k][idx]
        elif r < 0.90:
            seeds[k] = np.random.randint(0, 2, n).astype(np.int32)
        else:
            seeds[k] = best_model.copy()
            bsz = n // 4
            start = np.random.randint(0, n - bsz + 1)
            seeds[k][start:start + bsz] = 1 - seeds[k][start:start + bsz]
    return seeds


# ══════════════════════════════════════════════════════════════════════
#  BASIN CLUSTERING — Hamming distance (Numba)
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def basin_cluster_numba(cand_bin, cand_unsat, n, tau, max_clusters):
    K = cand_bin.shape[0]
    order = np.argsort(cand_unsat)
    centers = np.zeros(max_clusters, dtype=np.int32)
    n_clusters = 0

    for oi in range(K):
        i = order[oi]
        near = False
        for c in range(n_clusters):
            ci = centers[c]
            dist = 0
            for j in range(n):
                if cand_bin[i, j] != cand_bin[ci, j]:
                    dist += 1
                    if dist > tau:
                        break
            if dist <= tau:
                near = True
                break
        if not near and n_clusters < max_clusters:
            centers[n_clusters] = i
            n_clusters += 1

    return centers, n_clusters


# ══════════════════════════════════════════════════════════════════════
#  INSTANCE HELPER
# ══════════════════════════════════════════════════════════════════════

class InstanceHelper:
    def __init__(self, n, clauses):
        self.n = n
        self.m = len(clauses)
        vs_list = [vs for vs, ss in clauses]
        ss_list = [ss for vs, ss in clauses]
        self.vars_t = torch.tensor(vs_list, dtype=torch.long, device=device)
        self.signs_t = torch.tensor(ss_list, dtype=torch.float32, device=device)
        self.pos_mask = (self.signs_t > 0).long()
        self.clauses_v = np.array(vs_list, dtype=np.int32)
        self.clauses_s = np.array(ss_list, dtype=np.int32)


# ══════════════════════════════════════════════════════════════════════
#  BPR WORKER
# ══════════════════════════════════════════════════════════════════════

def bpr_worker(clauses_v, clauses_s, x_np, weight,
               max_flips, T_init, T_min, p_random, beta,
               chain_patience, branch_patience,
               cool_rate, stub_frac, weight_bump, weight_decay):
    sol, flips, n_chains, remaining = bpr_chain(
        clauses_v, clauses_s, x_np, weight,
        max_flips=max_flips, T_init=T_init, T_min=T_min,
        p_random=p_random, beta=beta,
        chain_patience=chain_patience, branch_patience=branch_patience,
        cool_rate=cool_rate, stub_frac=stub_frac,
        weight_bump=weight_bump, weight_decay=weight_decay
    )
    is_sat = check_sat(clauses_v, clauses_s, sol)
    return is_sat, flips, n_chains, sol


# ══════════════════════════════════════════════════════════════════════
#  FUSED GPU CANDIDATE EXTRACTION (H19++ optimisation)
#  Discretize + SAT eval + confidence computed ON GPU.
#  Only transfers tiny results: binary assignment + unsat + confidence.
#  K=16, P=250, n=1000: transfers ~16KB instead of ~16MB per batch.
# ══════════════════════════════════════════════════════════════════════

def _fused_extract_candidates(s_final, vars_t, signs_t, pos_mask):
    """
    GPU-side: discretize all particles, find best per engine, extract.
    s_final: (B, P, n) continuous particles
    Returns list of B dicts: {assignment, unsat, confidence}
    """
    B, P, n = s_final.shape
    with torch.no_grad():
        x_bin = (s_final > 0).long()  # (B, P, n)

        # Vectorized per-clause satisfaction (fully on GPU)
        m_cls = vars_t.shape[1]
        clause_sat = torch.zeros(B, P, m_cls, device=s_final.device, dtype=torch.bool)
        for j in range(3):
            idx_j = vars_t[:, :, j].unsqueeze(1).expand(B, P, m_cls)
            gathered = torch.gather(x_bin, 2, idx_j)
            pm_j = pos_mask[:, :, j].unsqueeze(1).expand(B, P, m_cls)
            clause_sat = clause_sat | (gathered == pm_j)
        n_sat = clause_sat.sum(dim=2)  # (B, P) — satisfied clauses per particle

        # Best particle per engine
        best_idx = n_sat.argmax(dim=1)  # (B,)
        best_n_sat = n_sat[torch.arange(B, device=s_final.device), best_idx]
        best_unsat = m_cls - best_n_sat

        # Extract best particles only (tiny transfer)
        best_assignments = x_bin[torch.arange(B, device=s_final.device), best_idx]
        best_conf = (1.0 - s_final[torch.arange(B, device=s_final.device), best_idx].abs())

        # Single CPU transfer
        assignments_np = best_assignments.cpu().numpy().astype(np.int32)
        unsats_np = best_unsat.cpu().numpy().astype(np.int32)
        conf_np = best_conf.cpu().numpy().astype(np.float64)

    results = []
    for b in range(B):
        results.append({
            'assignment': assignments_np[b],
            'unsat': int(unsats_np[b]),
            'confidence': conf_np[b],
        })
    return results


# ══════════════════════════════════════════════════════════════════════
#  RUN WAVE GRAVITY — batch K engines, fused extraction (H19++)
# ══════════════════════════════════════════════════════════════════════

def run_wave_gravity(unsolved_helpers, seeds_per_inst, n, m,
                     particles, steps, batch_size):
    """
    Run gravity for all engines across all unsolved instances.
    Fused GPU extraction: only transfers binary assignments + unsats.
    Returns: result[i] = list of K candidate dicts
    """
    N = len(unsolved_helpers)
    K = seeds_per_inst[0].shape[0]
    total = N * K

    flat_helper_idx = []
    flat_seeds = np.zeros((total, n), dtype=np.int32)
    for i in range(N):
        for k in range(K):
            flat_helper_idx.append(i)
            flat_seeds[i * K + k] = seeds_per_inst[i][k]

    all_candidates = [None] * total

    for bs in range(0, total, batch_size):
        be = min(bs + batch_size, total)
        B = be - bs

        vars_batch = torch.stack([
            unsolved_helpers[flat_helper_idx[e]].vars_t
            for e in range(bs, be)
        ])
        signs_batch = torch.stack([
            unsolved_helpers[flat_helper_idx[e]].signs_t
            for e in range(bs, be)
        ])
        pos_mask_batch = (signs_batch > 0).long()

        s_seeds = torch.tensor(
            flat_seeds[bs:be], dtype=torch.float32, device=device
        )
        s_seeds = s_seeds * 1.4 - 0.7
        s = s_seeds.unsqueeze(1).expand(B, particles, n).clone()
        s += torch.randn(B, particles, n, device=device) * 0.3
        s.clamp_(-0.9, 0.9)

        s_final = _gravity_core_h19(s, vars_batch, signs_batch, steps=steps)

        # Fused extraction on GPU
        batch_candidates = _fused_extract_candidates(
            s_final, vars_batch, signs_batch, pos_mask_batch
        )
        for b in range(B):
            all_candidates[bs + b] = batch_candidates[b]

        del s, s_final, vars_batch, signs_batch, pos_mask_batch
        torch.cuda.empty_cache() if device.type == 'cuda' else None

    result = []
    for i in range(N):
        result.append([all_candidates[i * K + k] for k in range(K)])
    return result


# ══════════════════════════════════════════════════════════════════════
#  PROCESS ONE INSTANCE IN A WAVE (filter → cluster → BPR)
# ══════════════════════════════════════════════════════════════════════

def process_instance_wave(helper, candidates_list, best_model,
                          best_unsat, n):
    """
    Full pipeline for one instance after gravity.
    H19++: candidates already extracted on GPU (fused extraction).
    Returns: (solved, new_best_model, new_best_unsat,
              n_basins, n_bpr_ran, gravity_only)
    """
    m = helper.m
    candidates = candidates_list

    # Check gravity-only solves
    for c in candidates:
        if c['unsat'] == 0:
            return True, c['assignment'], 0, 0, 0, True

    # Stage 1: reject if unsat > 0.20 × n
    threshold = int(UNSAT_REJECT_FRAC * n)
    candidates = [c for c in candidates if c['unsat'] <= threshold]

    if not candidates:
        return False, best_model, best_unsat, 0, 0, False

    # Stage 2: sort by unsat, keep top
    candidates.sort(key=lambda c: c['unsat'])
    candidates = candidates[:TOP_CANDIDATES]

    # Basin clustering
    cand_bin = np.array([c['assignment'] for c in candidates], dtype=np.int32)
    cand_unsat = np.array([c['unsat'] for c in candidates], dtype=np.int32)
    tau = int(HAMMING_TAU_FRAC * n)

    centers, n_clusters = basin_cluster_numba(
        cand_bin, cand_unsat, n, tau, MAX_BASINS
    )
    basin_indices = [int(centers[c]) for c in range(n_clusters)]

    # Selective BPR
    n_bpr_ran = 0
    solved = False
    new_best_model = best_model
    new_best_unsat = best_unsat

    bpr_tasks = []
    for bi in basin_indices:
        cand = candidates[bi]
        unsat_frac = cand['unsat'] / m

        run_prob = 0.0
        flip_budget = 0
        for (thresh, prob, budget) in BPR_TIERS:
            if unsat_frac < thresh:
                run_prob = prob
                flip_budget = budget
                break

        if np.random.random() < run_prob:
            bpr_tasks.append((bi, flip_budget))

    if not bpr_tasks:
        best_cand = candidates[basin_indices[0]] if basin_indices else None
        if best_cand and best_cand['unsat'] < new_best_unsat:
            new_best_model = best_cand['assignment']
            new_best_unsat = best_cand['unsat']
        return False, new_best_model, new_best_unsat, n_clusters, 0, False

    with ThreadPoolExecutor(max_workers=min(N_WORKERS, len(bpr_tasks))) as pool:
        futures = {}
        for (bi, flips) in bpr_tasks:
            cand = candidates[bi]
            f = pool.submit(
                bpr_worker, helper.clauses_v, helper.clauses_s,
                cand['assignment'].copy(), cand['confidence'],
                flips, 0.5, 0.01, 0.1, BPR_BETA,
                CHAIN_PATIENCE, BRANCH_PATIENCE,
                COOL_RATE, STUB_FRAC, WEIGHT_BUMP, WEIGHT_DECAY
            )
            futures[f] = bi
            n_bpr_ran += 1

        for f in as_completed(futures):
            is_sat, flips_used, n_chains, sol = f.result()
            sol_unsat = 0 if is_sat else count_unsat(
                helper.clauses_v, helper.clauses_s, sol
            )

            if is_sat:
                solved = True
                new_best_model = sol
                new_best_unsat = 0
            elif sol_unsat < new_best_unsat:
                new_best_unsat = sol_unsat
                new_best_model = sol.copy()

    return solved, new_best_model, new_best_unsat, n_clusters, n_bpr_ran, False


# ══════════════════════════════════════════════════════════════════════
#  COMPILE + WARMUP
# ══════════════════════════════════════════════════════════════════════

_wv = np.array([[0, 1, 2]], dtype=np.int32)
_ws = np.array([[1, -1, 1]], dtype=np.int32)
_wa = np.array([1, 0, 1], dtype=np.int32)
_ww = np.array([0.5, 0.3, 0.8], dtype=np.float64)
_ = bpr_chain(_wv, _ws, _wa, _ww, max_flips=100,
              chain_patience=20, branch_patience=5)
_ = check_sat(_wv, _ws, _wa)
_ = count_unsat(_wv, _ws, _wa)
_cb = np.array([[1, 0, 1], [0, 1, 0]], dtype=np.int32)
_cu = np.array([1, 2], dtype=np.int32)
_ = basin_cluster_numba(_cb, _cu, 3, 1, 2)

print(f'CPU cores: {os.cpu_count()}, BPR workers: {N_WORKERS}')
print()
print('✓ H19 — Wave-Based Multi-Engine SAT Solver with Basin Clustering')
print(f'  Device:  {device}')
if device.type == 'cuda':
    print(f'  GPU:     {torch.cuda.get_device_name()}')
    print(f'  VRAM:    {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print()
print(f'  K_BASE = {K_BASE} engines | P = {P_ENGINE} particles/engine')
print(f'  Steps = {STEPS_ENGINE} | Max waves = {MAX_WAVES}')
print(f'  Engine batch = {ENGINE_BATCH} | Max basins = {MAX_BASINS}')
print()
print('  Per-wave pipeline:')
print('    ┌─ 1. Generate K diverse seeds (5 types: local/inv/flip/rand/mask)')
print('    ├─ 2. Run K gravity engines (independent, GPU batch)')
print('    ├─ 3. Fused GPU extraction (discretize+eval+extract on GPU)')
print('    ├─ 4. Stage 1: reject if unsat/n > 20%')
print('    ├─ 5. Stage 2: rank by score, keep top 32')
print('    ├─ 6. Basin cluster (Hamming τ=15%n) → ≤6 distinct basins')
print('    ├─ 7. Selective BPR (budget: 60K/40K/20K/5K by proximity)')
print('    └─ 8. Update best_model + stagnation')
print()
print(f'  BPR tiers:')
for thresh, prob, budget in BPR_TIERS:
    pct = f'{thresh*100:.1f}%' if thresh < 1 else 'else'
    print(f'    unsat < {pct:>6}: p={prob:.0%}, {budget//1000}K flips')
print()
print(f'  Adaptive K: base={K_BASE} × (1 + stag/{STAG_SCALE}), cap={K_GPU_LIMIT}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H19 Experiment: Wave-Based Multi-Engine SAT Solver
# ══════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

ALPHAS   = [3.8, 4.0, 4.2]
NS       = [500, 750, 1000]
N_INST   = 50

# All baselines
h9b = {
    (3.8, 500): 94.0, (3.8, 750): 86.0, (3.8, 1000): 72.0,
    (4.0, 500): 44.0, (4.0, 750): 26.0, (4.0, 1000):  4.0,
    (4.2, 500):  8.0, (4.2, 750):  0.0, (4.2, 1000):  0.0,
}
h10b = {
    (3.8, 500): 98.0, (3.8, 750): 98.0, (3.8, 1000): 96.0,
    (4.0, 500): 72.0, (4.0, 750): 60.0, (4.0, 1000): 30.0,
    (4.2, 500): 10.0, (4.2, 750):  6.0, (4.2, 1000):  0.0,
}
h11b_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 68.0,  (4.0, 750): 76.0,  (4.0, 1000): 52.0,
    (4.2, 500):  8.0,  (4.2, 750):  2.0,  (4.2, 1000):  2.0,
}
h13_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 94.0,  (4.0, 750): 100.0, (4.0, 1000): 98.0,
    (4.2, 500): 30.0,  (4.2, 750): 32.0,  (4.2, 1000): 16.0,
}
h14_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 100.0, (4.0, 750): 100.0, (4.0, 1000): 98.0,
    (4.2, 500): 42.0,  (4.2, 750): 28.0,  (4.2, 1000): 20.0,
}
h15_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 100.0, (4.0, 750): 100.0, (4.0, 1000): 100.0,
    (4.2, 500): 54.0,  (4.2, 750): 36.0,  (4.2, 1000): 24.0,
}

results = {}
wave_stats = {}

print('=' * 140)
print('H19 — Wave-Based Multi-Engine SAT Solver with Basin Clustering')
print(f'  K_BASE={K_BASE} engines × {P_ENGINE} particles | '
      f'{STEPS_ENGINE} steps | Max {MAX_WAVES} waves')
print(f'  Stage 1: reject unsat>{UNSAT_REJECT_FRAC*100:.0f}%n | '
      f'Basin τ={HAMMING_TAU_FRAC*100:.0f}%n | Max {MAX_BASINS} basins')
print(f'  BPR: selective (60K/40K/20K/5K by proximity)')
print('=' * 140)
print(f"  {'α':>5} | {'n':>5} | {'Solved':>6} | {'GravO':>5} | "
      f"{'Waves':>5} | {'Basins':>6} | {'BPRs':>5} | "
      f"{'H15':>5} | {'H14':>5} | {'H13':>5} | "
      f"{'ΔvsH15':>7} | Time")
print('  ' + '-' * 120)

for alpha in ALPHAS:
    for n_var in NS:
        m_cls = int(alpha * n_var)
        t0 = time.time()

        # ═══════════════════════════════════════════════════════════
        # Generate instances + helpers
        # ═══════════════════════════════════════════════════════════
        all_instances = [generate_3sat_instance(n_var, m_cls) for _ in range(N_INST)]
        helpers = [InstanceHelper(n_var, all_instances[i]) for i in range(N_INST)]

        # Per-instance tracking
        inst_solved = [False] * N_INST
        inst_wave_solved = [0] * N_INST
        inst_best_model = [None] * N_INST
        inst_best_unsat = [m_cls] * N_INST
        inst_gravity_only = [False] * N_INST

        # Global wave stats
        total_basins_found = 0
        total_bpr_ran = 0
        total_grav_only = 0
        stagnation = 0
        waves_used = 0

        per_wave_solves = []

        # ═══════════════════════════════════════════════════════════
        # WAVE LOOP
        # ═══════════════════════════════════════════════════════════
        for wave in range(1, MAX_WAVES + 1):
            unsolved_idx = [i for i in range(N_INST) if not inst_solved[i]]
            if not unsolved_idx:
                break

            waves_used = wave
            t_wave = time.time()

            # Adaptive K
            K = K_BASE * (1 + stagnation // STAG_SCALE)
            K = min(K, K_GPU_LIMIT)
            N_unsolved = len(unsolved_idx)

            print(f'    α={alpha}, n={n_var}: W{wave} — '
                  f'{N_unsolved} unsolved, K={K} engines, '
                  f'stag={stagnation}')

            # ── Generate diverse seeds ──
            seeds_list = []
            unsolved_helpers = []
            for ui in unsolved_idx:
                seeds = generate_diverse_seeds(
                    K, n_var, inst_best_model[ui]
                )
                seeds_list.append(seeds)
                unsolved_helpers.append(helpers[ui])

            # ── Run gravity engines (batched on GPU) ──
            t_grav = time.time()
            engine_results = run_wave_gravity(
                unsolved_helpers, seeds_list,
                n_var, m_cls,
                particles=P_ENGINE,
                steps=STEPS_ENGINE,
                batch_size=ENGINE_BATCH
            )
            t_grav = time.time() - t_grav

            # ── Process each instance: filter → cluster → BPR ──
            wave_new_solves = 0
            wave_basins = 0
            wave_bpr = 0
            wave_improved = 0

            for ui_idx, orig_idx in enumerate(unsolved_idx):
                solved, new_model, new_unsat, n_basins, n_bpr, grav_only = \
                    process_instance_wave(
                        helpers[orig_idx],
                        engine_results[ui_idx],
                        inst_best_model[orig_idx],
                        inst_best_unsat[orig_idx],
                        n_var
                    )

                wave_basins += n_basins
                wave_bpr += n_bpr

                if solved:
                    inst_solved[orig_idx] = True
                    inst_wave_solved[orig_idx] = wave
                    inst_best_model[orig_idx] = new_model
                    inst_best_unsat[orig_idx] = 0
                    wave_new_solves += 1
                    if grav_only:
                        inst_gravity_only[orig_idx] = True
                        total_grav_only += 1
                else:
                    if new_unsat < inst_best_unsat[orig_idx]:
                        inst_best_model[orig_idx] = new_model
                        inst_best_unsat[orig_idx] = new_unsat
                        wave_improved += 1
                    else:
                        inst_best_model[orig_idx] = new_model if new_model is not None else inst_best_model[orig_idx]

            total_basins_found += wave_basins
            total_bpr_ran += wave_bpr

            total_solved = sum(inst_solved)
            t_wave_end = time.time() - t_wave
            per_wave_solves.append(wave_new_solves)

            print(f'      → W{wave}: +{wave_new_solves} solved '
                  f'({total_solved}/{N_INST}), '
                  f'{wave_basins} basins, {wave_bpr} BPRs, '
                  f'{wave_improved} improved | '
                  f'grav={t_grav:.0f}s, total={t_wave_end:.0f}s')

            # Stagnation update
            if wave_new_solves == 0 and wave_improved == 0:
                stagnation += 1
            else:
                stagnation = 0

            if total_solved == N_INST:
                break

        # ═══════════════════════════════════════════════════════════
        # RESULTS for this (α, n)
        # ═══════════════════════════════════════════════════════════
        elapsed = time.time() - t0
        solved_pct = sum(inst_solved) / N_INST * 100
        results[(alpha, n_var)] = solved_pct

        h15e = h15_ens[(alpha, n_var)]
        h14e = h14_ens[(alpha, n_var)]
        h13e = h13_ens[(alpha, n_var)]
        delta = solved_pct - h15e

        # Wave analysis for this config
        ws = {
            'waves_used': waves_used,
            'per_wave': per_wave_solves,
            'basins': total_basins_found,
            'bpr_ran': total_bpr_ran,
            'grav_only': total_grav_only,
            'wave_solved': inst_wave_solved[:],
            'unsolved_unsats': [inst_best_unsat[i] for i in range(N_INST)
                                if not inst_solved[i]],
        }
        wave_stats[(alpha, n_var)] = ws

        tag = '★' if solved_pct >= 95 else ('▲' if delta > 2 else
              ('≈' if abs(delta) <= 2 else '▼'))

        print(f'  {alpha:5.1f} | {n_var:5d} | {solved_pct:5.1f}% | '
              f'{total_grav_only:5d} | {waves_used:5d} | '
              f'{total_basins_found:6d} | {total_bpr_ran:5d} | '
              f'{h15e:4.0f}% | {h14e:4.0f}% | {h13e:4.0f}% | '
              f'{delta:+6.1f}% | {elapsed:.0f}s {tag}')
    print('  ' + '-' * 120)


# ══════════════════════════════════════════════════════════════════════
#  FULL COMPARISON TABLE
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 140)
print('FULL COMPARISON — H9b → H10b → H11b → H13 → H14 → H15 → H19')
print('=' * 140)
print(f"  {'α':>5} | {'n':>5} | {'H9b':>5} | {'H10b':>5} | "
      f"{'H11bE':>6} | {'H13E':>5} | {'H14E':>5} | "
      f"{'H15':>5} | {'H19':>5} | {'ΔvsH15':>7}")
print('  ' + '-' * 90)
for alpha in ALPHAS:
    for n_var in NS:
        h19 = results[(alpha, n_var)]
        h15e = h15_ens[(alpha, n_var)]
        delta = h19 - h15e
        print(f'  {alpha:5.1f} | {n_var:5d} | '
              f'{h9b[(alpha,n_var)]:4.0f}% | '
              f'{h10b[(alpha,n_var)]:4.0f}% | '
              f'{h11b_ens[(alpha,n_var)]:5.1f}% | '
              f'{h13_ens[(alpha,n_var)]:4.0f}% | '
              f'{h14_ens[(alpha,n_var)]:4.0f}% | '
              f'{h15e:4.0f}% | {h19:4.0f}% | {delta:+6.1f}%')
    print('  ' + '-' * 90)


# ══════════════════════════════════════════════════════════════════════
#  WAVE-BY-WAVE ANALYSIS
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 140)
print('WAVE ANALYSIS — which waves solved instances?')
print('=' * 140)

hdr = f"  {'α':>5} | {'n':>5}"
for w in range(1, MAX_WAVES + 1):
    hdr += f" | {'W'+str(w):>4}"
hdr += f" | {'Fail':>4} | {'Basins':>6} | {'BPRs':>5} | {'GravO':>5} | Interpretation"
print(hdr)
print('  ' + '-' * 130)

for alpha in ALPHAS:
    for n_var in NS:
        ws = wave_stats[(alpha, n_var)]
        row = f'  {alpha:5.1f} | {n_var:5d}'
        wave_counts = {}
        for w_s in ws['wave_solved']:
            if w_s > 0:
                wave_counts[w_s] = wave_counts.get(w_s, 0) + 1
        for w in range(1, MAX_WAVES + 1):
            row += f' | {wave_counts.get(w, 0):4d}'
        n_fail = len(ws['unsolved_unsats'])
        row += f' | {n_fail:4d} | {ws["basins"]:6d} | {ws["bpr_ran"]:5d} | {ws["grav_only"]:5d}'

        # Interpretation
        total_solved = N_INST - n_fail
        w1 = wave_counts.get(1, 0)
        later = total_solved - w1
        if n_fail == 0:
            if w1 == N_INST:
                interp = 'All W1'
            else:
                interp = f'W1={w1}, later waves rescued +{later}'
        else:
            if later > 0:
                interp = f'W1={w1}, +{later} rescued, {n_fail} hard'
            else:
                interp = f'W1={w1} only, {n_fail} hard'
        row += f' | {interp}'
        print(row)
    print('  ' + '-' * 130)


# ══════════════════════════════════════════════════════════════════════
#  UNSOLVED ANALYSIS
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 100)
print('UNSOLVED INSTANCE ANALYSIS')
print('=' * 100)
for alpha in ALPHAS:
    for n_var in NS:
        ws = wave_stats[(alpha, n_var)]
        unsats = ws['unsolved_unsats']
        if not unsats:
            continue
        print(f'  α={alpha}, n={n_var}: {len(unsats)} unsolved | '
              f'unsat: min={min(unsats)}, median={np.median(unsats):.0f}, '
              f'max={max(unsats)}, mean={np.mean(unsats):.1f}')


# ══════════════════════════════════════════════════════════════════════
#  UNSAT ESTIMATION (P(miss) heuristic)
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 100)
print('BASIN EXPLORATION — UNSAT probability estimate')
print('=' * 100)
for alpha in ALPHAS:
    for n_var in NS:
        ws = wave_stats[(alpha, n_var)]
        total_solved = N_INST - len(ws['unsolved_unsats'])
        if total_solved == N_INST or ws['basins'] == 0:
            continue
        p_hat = total_solved / N_INST
        B_basins = ws['basins']
        if p_hat > 0:
            p_miss = (1 - p_hat) ** B_basins
        else:
            p_miss = 1.0
        print(f'  α={alpha}, n={n_var}: basins={B_basins}, '
              f'p̂={p_hat:.2f}, P(miss)={(1-p_hat):.3f}^{B_basins} = '
              f'{p_miss:.2e}')


# ══════════════════════════════════════════════════════════════════════
#  CHARTS
# ══════════════════════════════════════════════════════════════════════
print('\nGenerating charts...\n')

fig, axes = plt.subplots(1, 3, figsize=(24, 7))
w = 0.10
colors = ['#e74c3c', '#3498db', '#f39c12', '#2ecc71',
          '#8e44ad', '#1abc9c', '#e67e22', '#c0392b']
labels_c = ['H9b', 'H10b', 'H11b', 'H13', 'H14', 'H15', 'H19']

for i, n_var in enumerate(NS):
    ax = axes[i]
    x = np.arange(len(ALPHAS))

    bars_data = [
        [h9b[(a, n_var)] for a in ALPHAS],
        [h10b[(a, n_var)] for a in ALPHAS],
        [h11b_ens[(a, n_var)] for a in ALPHAS],
        [h13_ens[(a, n_var)] for a in ALPHAS],
        [h14_ens[(a, n_var)] for a in ALPHAS],
        [h15_ens[(a, n_var)] for a in ALPHAS],
        [results[(a, n_var)] for a in ALPHAS],
    ]

    for k, (label, vals) in enumerate(zip(labels_c, bars_data)):
        offset = (k - 3) * w
        ax.bar(x + offset, vals, w, label=label, color=colors[k], alpha=0.85)

    ax.set_xticks(list(x))
    ax.set_xticklabels([str(a) for a in ALPHAS])
    ax.set_xlabel('α (clause ratio)')
    ax.set_ylabel('Solve Rate %')
    ax.set_title(f'n = {n_var}')
    ax.set_ylim(0, 105)
    ax.legend(fontsize=5.5, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('H19: Wave-Based Multi-Engine SAT Solver\n'
             f'K={K_BASE} engines × {P_ENGINE} particles | '
             f'{STEPS_ENGINE} steps | Basin clustering (τ={HAMMING_TAU_FRAC*100:.0f}%n) | '
             f'Selective BPR | Max {MAX_WAVES} waves',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('h19_wave_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: h19_wave_results.png')

# ── Effective budget summary ──
print(f'\nEffective budget per instance (max {MAX_WAVES} waves):')
max_bpr_budget = max(b for _, _, b in BPR_TIERS) * MAX_BASINS * MAX_WAVES
print(f'  Gravity: {MAX_WAVES} × {STEPS_ENGINE} steps × K engines')
print(f'  BPR max: {MAX_BASINS} basins × {MAX_WAVES} waves × '
      f'{max(b for _,_,b in BPR_TIERS)//1000}K = '
      f'{max_bpr_budget//1000}K flips (theoretical max)')
print(f'  Actual BPR much less due to selective filtering')